
# Roxy notebook example: N-terminal and C-terminal descriptors

This notebook is a **reference implementation example** for the **terminal descriptor family** in Roxy.

Terminal descriptors are highly useful because many biological signals are enriched near the **N-terminus** or **C-terminus** rather than being uniformly distributed across the full sequence.

## Covered outputs

This notebook implements terminal descriptors for configurable windows such as:

- N5, N10, N20
- C5, C10, C20

For each terminal window, the notebook computes:

- terminal length
- amino acid composition
- grouped residue composition
- hydrophobicity mean
- polarity mean
- aromaticity fraction
- charge-related fractions
- Shannon entropy
- simple terminal-specific balances

The notebook is written as a **clean teaching implementation** so it can later be migrated into the real Roxy package.


In [1]:

from collections import Counter

import numpy as np
import pandas as pd


## Demo dataset

In [2]:

df_demo = pd.DataFrame(
    {
        "sequence_id": [
            "term_1",
            "term_2",
            "term_3",
            "term_4",
            "term_5",
            "term_6",
        ],
        "sequence": [
            "MKWVTFISLLFLFSSAYSRGVFRR",
            "GGGGGGGGGGGGGGG",
            "KRRKRRKRRKRRDDDDEE",
            "ACDEFGHIKLMNPQRSTVWY",
            "PPPPGSSSSSTTTTNNQQQ",
            "MSTNPKPQRITLKDGNKVELV",
        ],
        "label": ["A", "B", "A", "B", "A", "B"],
    }
)

df_demo


,sequence_id,sequence,label
0,term_1,MKWVTFISLLFLFSSAYSRGVFRR,A
1,term_2,GGGGGGGGGGGGGGG,B
2,term_3,KRRKRRKRRKRRDDDDEE,A
3,term_4,ACDEFGHIKLMNPQRSTVWY,B
4,term_5,PPPPGSSSSSTTTTNNQQQ,A
5,term_6,MSTNPKPQRITLKDGNKVELV,B


## Constants

In [3]:

STANDARD_AA = list("ACDEFGHIKLMNPQRSTVWY")
STANDARD_AA_SET = set(STANDARD_AA)

HYDROPATHY = {
    "A": 1.8, "C": 2.5, "D": -3.5, "E": -3.5, "F": 2.8,
    "G": -0.4, "H": -3.2, "I": 4.5, "K": -3.9, "L": 3.8,
    "M": 1.9, "N": -3.5, "P": -1.6, "Q": -3.5, "R": -4.5,
    "S": -0.8, "T": -0.7, "V": 4.2, "W": -0.9, "Y": -1.3,
}

POLARITY = {
    "A": 8.1, "C": 5.5, "D": 13.0, "E": 12.3, "F": 5.2,
    "G": 9.0, "H": 10.4, "I": 5.2, "K": 11.3, "L": 4.9,
    "M": 5.7, "N": 11.6, "P": 8.0, "Q": 10.5, "R": 10.5,
    "S": 9.2, "T": 8.6, "V": 5.9, "W": 5.4, "Y": 6.2,
}

AA_GROUPS = {
    "positive": set("KRH"),
    "negative": set("DE"),
    "charged": set("KRHDE"),
    "polar": set("STNQCYWHKRDE"),
    "nonpolar": set("AVLIMFGP"),
    "aromatic": set("FWYH"),
    "aliphatic": set("AVLIM"),
    "tiny": set("AGCS"),
    "small": set("AGCSTVPDN"),
    "branched": set("VILT"),
    "hydrophobic": set("AVLIMFWCY"),
    "hydrophilic": set("RNDQEHKST"),
}


## Helper functions

In [4]:

def clean_sequence(seq: str) -> str:
    """Keep only the 20 standard amino acids."""
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper().replace("*", "")
    return "".join([aa for aa in seq if aa in STANDARD_AA_SET])


def get_terminal_window(seq: str, side: str, window_size: int) -> str:
    """Return N- or C-terminal subsequence. If the sequence is shorter, return the full sequence."""
    seq = clean_sequence(seq)
    if side == "N":
        return seq[:window_size]
    if side == "C":
        return seq[-window_size:]
    raise ValueError("side must be 'N' or 'C'")


def shannon_entropy(seq: str) -> float:
    if len(seq) == 0:
        return np.nan
    counts = Counter(seq)
    probs = np.array([count / len(seq) for count in counts.values()], dtype=float)
    return float(-(probs * np.log2(probs)).sum())


def scale_mean(seq: str, scale: dict) -> float:
    if len(seq) == 0:
        return np.nan
    values = [scale[aa] for aa in seq]
    return float(np.mean(values))


def fraction_from_group(seq: str, aa_group) -> float:
    if len(seq) == 0:
        return np.nan
    return sum(aa in aa_group for aa in seq) / len(seq)


def safe_ratio(num: float, den: float) -> float:
    if den == 0:
        return np.nan
    return num / den


def amino_acid_frequencies(seq: str, prefix: str) -> dict:
    if len(seq) == 0:
        return {f"{prefix}_aac_{aa}": np.nan for aa in STANDARD_AA}
    counts = Counter(seq)
    return {f"{prefix}_aac_{aa}": counts.get(aa, 0) / len(seq) for aa in STANDARD_AA}


def grouped_fractions(seq: str, prefix: str) -> dict:
    return {f"{prefix}_{name}_frac": fraction_from_group(seq, group) for name, group in AA_GROUPS.items()}


## Core terminal descriptor function

In [5]:

def terminal_descriptors(seq: str, window_sizes=(5, 10, 20)) -> dict:
    seq = clean_sequence(seq)
    out = {
        "term_length": len(seq),
    }

    for window in window_sizes:
        for side in ("N", "C"):
            terminal_seq = get_terminal_window(seq, side=side, window_size=window)
            prefix = f"{side.lower()}term{window}"

            positive_frac = fraction_from_group(terminal_seq, AA_GROUPS["positive"])
            negative_frac = fraction_from_group(terminal_seq, AA_GROUPS["negative"])
            hydrophobic_frac = fraction_from_group(terminal_seq, AA_GROUPS["hydrophobic"])
            hydrophilic_frac = fraction_from_group(terminal_seq, AA_GROUPS["hydrophilic"])

            out[f"{prefix}_length"] = len(terminal_seq)
            out[f"{prefix}_hydropathy_mean"] = scale_mean(terminal_seq, HYDROPATHY)
            out[f"{prefix}_polarity_mean"] = scale_mean(terminal_seq, POLARITY)
            out[f"{prefix}_entropy"] = shannon_entropy(terminal_seq)
            out[f"{prefix}_positive_frac"] = positive_frac
            out[f"{prefix}_negative_frac"] = negative_frac
            out[f"{prefix}_charged_frac"] = fraction_from_group(terminal_seq, AA_GROUPS["charged"])
            out[f"{prefix}_aromatic_frac"] = fraction_from_group(terminal_seq, AA_GROUPS["aromatic"])
            out[f"{prefix}_hydrophobic_frac"] = hydrophobic_frac
            out[f"{prefix}_hydrophilic_frac"] = hydrophilic_frac
            out[f"{prefix}_polar_frac"] = fraction_from_group(terminal_seq, AA_GROUPS["polar"])
            out[f"{prefix}_nonpolar_frac"] = fraction_from_group(terminal_seq, AA_GROUPS["nonpolar"])
            out[f"{prefix}_positive_negative_balance"] = (
                positive_frac - negative_frac if not np.isnan(positive_frac) and not np.isnan(negative_frac) else np.nan
            )
            out[f"{prefix}_hydrophobic_hydrophilic_balance"] = (
                hydrophobic_frac - hydrophilic_frac if not np.isnan(hydrophobic_frac) and not np.isnan(hydrophilic_frac) else np.nan
            )
            out[f"{prefix}_positive_negative_ratio"] = safe_ratio(
                sum(aa in AA_GROUPS["positive"] for aa in terminal_seq),
                sum(aa in AA_GROUPS["negative"] for aa in terminal_seq),
            )

            out.update(amino_acid_frequencies(terminal_seq, prefix=prefix))
            out.update(grouped_fractions(terminal_seq, prefix=prefix))

    return out


## Functional usage on one sequence

In [6]:

example = terminal_descriptors(df_demo.loc[0, "sequence"], window_sizes=(5, 10))
list(example.items())[:20]


[('term_length', 24),
 ('nterm5_length', 5),
 ('nterm5_hydropathy_mean', 0.12000000000000006),
 ('nterm5_polarity_mean', 7.38),
 ('nterm5_entropy', 2.321928094887362),
 ('nterm5_positive_frac', 0.2),
 ('nterm5_negative_frac', 0.0),
 ('nterm5_charged_frac', 0.2),
 ('nterm5_aromatic_frac', 0.2),
 ('nterm5_hydrophobic_frac', 0.6),
 ('nterm5_hydrophilic_frac', 0.4),
 ('nterm5_polar_frac', 0.6),
 ('nterm5_nonpolar_frac', 0.4),
 ('nterm5_positive_negative_balance', 0.2),
 ('nterm5_hydrophobic_hydrophilic_balance', 0.19999999999999996),
 ('nterm5_positive_negative_ratio', nan),
 ('nterm5_aac_A', 0.0),
 ('nterm5_aac_C', 0.0),
 ('nterm5_aac_D', 0.0),
 ('nterm5_aac_E', 0.0)]

## Apply terminal descriptors to the full dataset

In [7]:

df_term = pd.concat(
    [
        df_demo,
        df_demo["sequence"].apply(lambda x: terminal_descriptors(x, window_sizes=(5, 10, 20))).apply(pd.Series),
    ],
    axis=1,
)

df_term.head()


,sequence_id,sequence,label,term_length,nterm5_length,nterm5_hydropathy_mean,nterm5_polarity_mean,nterm5_entropy,nterm5_positive_frac,nterm5_negative_frac,...,cterm20_aac_R,cterm20_aac_S,cterm20_aac_T,cterm20_aac_V,cterm20_aac_W,cterm20_aac_Y,cterm20_aliphatic_frac,cterm20_tiny_frac,cterm20_small_frac,cterm20_branched_frac
0,term_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24.0,5.0,0.12,7.38,2.321928,0.2,0.0,...,0.150000,0.200000,0.050000,0.05,0.00,0.05,0.30,0.300000,0.400000,0.300000
1,term_2,GGGGGGGGGGGGGGG,B,15.0,5.0,-0.40,9.00,-0.000000,0.0,0.0,...,0.000000,0.000000,0.000000,0.00,0.00,0.00,0.00,1.000000,1.000000,0.000000
2,term_3,KRRKRRKRRKRRDDDDEE,A,18.0,5.0,-4.26,10.82,0.970951,1.0,0.0,...,0.444444,0.000000,0.000000,0.00,0.00,0.00,0.00,0.000000,0.222222,0.000000
3,term_4,ACDEFGHIKLMNPQRSTVWY,B,20.0,5.0,0.02,8.82,2.321928,0.0,0.4,...,0.050000,0.050000,0.050000,0.05,0.05,0.05,0.25,0.200000,0.450000,0.200000
4,term_5,PPPPGSSSSSTTTTNNQQQ,A,19.0,5.0,-1.36,8.20,0.721928,0.0,0.0,...,0.000000,0.263158,0.210526,0.00,0.00,0.00,0.00,0.315789,0.842105,0.210526


## Inspect terminal descriptor groups

In [8]:

nterm10_cols = [c for c in df_term.columns if c.startswith("nterm10_")]
cterm10_cols = [c for c in df_term.columns if c.startswith("cterm10_")]

len(nterm10_cols), len(cterm10_cols)


(39, 39)

In [9]:

df_term[["sequence_id", "nterm10_hydropathy_mean", "cterm10_hydropathy_mean",
         "nterm10_positive_frac", "cterm10_positive_frac",
         "nterm10_entropy", "cterm10_entropy"]]


,sequence_id,nterm10_hydropathy_mean,cterm10_hydropathy_mean,nterm10_positive_frac,cterm10_positive_frac,nterm10_entropy,cterm10_entropy
0,term_1,1.47,-0.80,0.1,0.3,3.121928,2.646439
1,term_2,-0.40,-0.40,0.0,0.0,-0.000000,-0.000000
2,term_3,-4.26,-3.84,1.0,0.4,0.970951,1.846439
3,term_4,0.09,-1.07,0.2,0.1,3.321928,3.321928
4,term_5,-1.08,-2.11,0.0,0.0,1.360964,1.846439
5,term_6,-1.37,-0.27,0.2,0.2,3.121928,2.721928


## Compare N-terminal vs C-terminal summaries

In [10]:

comparison = pd.DataFrame(
    {
        "sequence_id": df_term["sequence_id"],
        "nterm10_hydropathy_mean": df_term["nterm10_hydropathy_mean"],
        "cterm10_hydropathy_mean": df_term["cterm10_hydropathy_mean"],
        "nterm10_positive_frac": df_term["nterm10_positive_frac"],
        "cterm10_positive_frac": df_term["cterm10_positive_frac"],
        "nterm10_entropy": df_term["nterm10_entropy"],
        "cterm10_entropy": df_term["cterm10_entropy"],
    }
)

comparison


,sequence_id,nterm10_hydropathy_mean,cterm10_hydropathy_mean,nterm10_positive_frac,cterm10_positive_frac,nterm10_entropy,cterm10_entropy
0,term_1,1.47,-0.80,0.1,0.3,3.121928,2.646439
1,term_2,-0.40,-0.40,0.0,0.0,-0.000000,-0.000000
2,term_3,-4.26,-3.84,1.0,0.4,0.970951,1.846439
3,term_4,0.09,-1.07,0.2,0.1,3.321928,3.321928
4,term_5,-1.08,-2.11,0.0,0.0,1.360964,1.846439
5,term_6,-1.37,-0.27,0.2,0.2,3.121928,2.721928


## Dataset-level summary for terminal balance descriptors

In [11]:

term_balance_cols = [
    c for c in df_term.columns
    if c.endswith("_positive_negative_balance") or c.endswith("_hydrophobic_hydrophilic_balance")
]

terminal_summary = (
    df_term[term_balance_cols]
    .mean(axis=0)
    .sort_values(ascending=False)
    .rename("mean_value")
    .reset_index()
    .rename(columns={"index": "descriptor"})
)

terminal_summary.head(12)


,descriptor,mean_value
0,nterm10_positive_negative_balance,0.216667
1,nterm5_positive_negative_balance,0.133333
2,cterm20_positive_negative_balance,0.105556
3,nterm20_positive_negative_balance,0.097222
4,cterm10_positive_negative_balance,0.033333
5,cterm5_positive_negative_balance,-0.100000
6,nterm5_hydrophobic_hydrophilic_balance,-0.166667
7,nterm10_hydrophobic_hydrophilic_balance,-0.233333
8,cterm5_hydrophobic_hydrophilic_balance,-0.266667
9,nterm20_hydrophobic_hydrophilic_balance,-0.306140


## Sanity checks

In [12]:

assert "nterm5_length" in df_term.columns
assert "cterm10_hydropathy_mean" in df_term.columns
assert "nterm20_entropy" in df_term.columns
assert "cterm5_positive_frac" in df_term.columns
assert "nterm10_aac_A" in df_term.columns
assert "cterm10_aromatic_frac" in df_term.columns
assert df_term["term_length"].min() > 0

print("Terminal descriptor checks passed.")
print(f"Total terminal descriptor columns: {sum(c.startswith(('nterm', 'cterm')) for c in df_term.columns)}")


Terminal descriptor checks passed.
Total terminal descriptor columns: 234


## Class-style implementation closer to the real package

In [13]:

class TerminalDescriptors:
    """Example class-style terminal descriptor implementation for later migration into Roxy."""

    def __init__(self, window_sizes=(5, 10, 20)):
        self.window_sizes = window_sizes

    def transform_sequence(self, seq: str) -> dict:
        return terminal_descriptors(seq, window_sizes=self.window_sizes)

    def transform(self, sequences) -> pd.DataFrame:
        return pd.DataFrame([self.transform_sequence(seq) for seq in sequences])


term_transformer = TerminalDescriptors(window_sizes=(5, 10, 20))
term_matrix = term_transformer.transform(df_demo["sequence"].tolist())
term_matrix.head()


,term_length,nterm5_length,nterm5_hydropathy_mean,nterm5_polarity_mean,nterm5_entropy,nterm5_positive_frac,nterm5_negative_frac,nterm5_charged_frac,nterm5_aromatic_frac,nterm5_hydrophobic_frac,...,cterm20_aac_R,cterm20_aac_S,cterm20_aac_T,cterm20_aac_V,cterm20_aac_W,cterm20_aac_Y,cterm20_aliphatic_frac,cterm20_tiny_frac,cterm20_small_frac,cterm20_branched_frac
0,24,5,0.12,7.38,2.321928,0.2,0.0,0.2,0.2,0.6,...,0.150000,0.200000,0.050000,0.05,0.00,0.05,0.30,0.300000,0.400000,0.300000
1,15,5,-0.40,9.00,-0.000000,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.00,0.00,0.00,0.00,1.000000,1.000000,0.000000
2,18,5,-4.26,10.82,0.970951,1.0,0.0,1.0,0.0,0.0,...,0.444444,0.000000,0.000000,0.00,0.00,0.00,0.00,0.000000,0.222222,0.000000
3,20,5,0.02,8.82,2.321928,0.0,0.4,0.4,0.2,0.6,...,0.050000,0.050000,0.050000,0.05,0.05,0.05,0.25,0.200000,0.450000,0.200000
4,19,5,-1.36,8.20,0.721928,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.263158,0.210526,0.00,0.00,0.00,0.00,0.315789,0.842105,0.210526


## Merge transformer output back to the dataset

In [14]:

df_term_class = pd.concat([df_demo, term_matrix], axis=1)
df_term_class.head()


,sequence_id,sequence,label,term_length,nterm5_length,nterm5_hydropathy_mean,nterm5_polarity_mean,nterm5_entropy,nterm5_positive_frac,nterm5_negative_frac,...,cterm20_aac_R,cterm20_aac_S,cterm20_aac_T,cterm20_aac_V,cterm20_aac_W,cterm20_aac_Y,cterm20_aliphatic_frac,cterm20_tiny_frac,cterm20_small_frac,cterm20_branched_frac
0,term_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24,5,0.12,7.38,2.321928,0.2,0.0,...,0.150000,0.200000,0.050000,0.05,0.00,0.05,0.30,0.300000,0.400000,0.300000
1,term_2,GGGGGGGGGGGGGGG,B,15,5,-0.40,9.00,-0.000000,0.0,0.0,...,0.000000,0.000000,0.000000,0.00,0.00,0.00,0.00,1.000000,1.000000,0.000000
2,term_3,KRRKRRKRRKRRDDDDEE,A,18,5,-4.26,10.82,0.970951,1.0,0.0,...,0.444444,0.000000,0.000000,0.00,0.00,0.00,0.00,0.000000,0.222222,0.000000
3,term_4,ACDEFGHIKLMNPQRSTVWY,B,20,5,0.02,8.82,2.321928,0.0,0.4,...,0.050000,0.050000,0.050000,0.05,0.05,0.05,0.25,0.200000,0.450000,0.200000
4,term_5,PPPPGSSSSSTTTTNNQQQ,A,19,5,-1.36,8.20,0.721928,0.0,0.0,...,0.000000,0.263158,0.210526,0.00,0.00,0.00,0.00,0.315789,0.842105,0.210526



## Suggested next refactor into the package

A clean migration path into Roxy would be:

- move terminal helper logic into `roxy/sequence/terminal.py`
- keep amino-acid constants and scales in `roxy/core/constants.py`
- expose a class such as `TerminalDescriptors`
- allow configurable:
  - window sizes
  - sides (N only, C only, or both)
  - descriptor subsets (AAC only, grouped only, all)
- add tests for:
  - sequences shorter than the requested terminal window
  - empty sequences
  - strongly charged termini
  - hydrophobic N-termini vs hydrophilic C-termini
  - lower-case input and invalid characters


## Optional export

In [ ]:
# df_term.to_csv("demo_terminal_descriptors.csv", index=False)
